# 05 iFinD 全量数据补齐候选方案

本 notebook 只生成候选数据文件和校验报告，默认不覆盖正式 raw 文件。目标区间：2014-01-01 至 2026-04-17；严格复现优先级：历史指数成分权重、历史行业分类、财务公告日/PIT。

## Section 0: 环境和参数

In [3]:
!pip install iFinDPy

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
ERROR: Could not find a version that satisfies the requirement iFinDPy (from versions: none)
ERROR: No matching distribution found for iFinDPy


In [1]:
from pathlib import Path
import os, time, json, math, shutil, warnings
from datetime import datetime
import pandas as pd
import numpy as np
import duckdb

PROJECT_ROOT = Path('/Users/shenboheng/Documents/ClaudeCode/ETF行业轮动/etf_style_rotation 3')
RAW = PROJECT_ROOT / 'data' / 'raw'
START_DATE = '2014-01-01'
END_DATE = '2026-04-17'
OVERWRITE_CANONICAL = False

FETCH_STOCK_DAILY = True
FETCH_STOCK_INDUSTRY = True
FETCH_INDEX_CONSTITUENTS = True
FETCH_FINANCIALS = True
FETCH_ETF = True
FETCH_MACRO = True

RATE_LIMIT_SLEEP = 0.35
MAX_RETRIES = 3
STOCK_BATCH_SIZE = 300
FIELD_BATCH_SIZE = 8
DATE_CHUNK_YEARS = 1
PART_DIR = RAW / 'ifind_candidate_parts'
PART_DIR.mkdir(exist_ok=True)
FAIL_LOG = RAW / 'ifind_fetch_failures.csv'

try:
    from iFinDPy import THS_iFinDLogin, THS_iFinDLogout, THS_BasicData, THS_HQ, THS_EDB, THS_DR, THS_DS
    IFIND_AVAILABLE = True
except Exception as e:
    IFIND_AVAILABLE = False
    IFIND_IMPORT_ERROR = repr(e)

def ifind_login_check():
    if not IFIND_AVAILABLE:
        print('iFinD Python 包不可用:', IFIND_IMPORT_ERROR)
        return False
    user = os.getenv('IFIND_USER') or os.getenv('THS_USER')
    password = os.getenv('IFIND_PASSWORD') or os.getenv('THS_PASSWORD')
    if not user or not password:
        print('未发现 IFIND_USER/IFIND_PASSWORD 环境变量；请先设置后再抓取。')
        return False
    ret = THS_iFinDLogin(user, password)
    print('iFinD login return:', ret)
    return str(ret).strip() in {'0', '0.0', 'success', 'True'}

IFIND_LOGGED_IN = ifind_login_check()

iFinD Python 包不可用: ModuleNotFoundError("No module named 'iFinDPy'")


## Section 1: 研报需求清单

In [ ]:
required_inventory = pd.read_csv(RAW / 'ifind_required_data_inventory.csv')
required_inventory.to_csv(RAW / 'ifind_required_data_inventory.csv', index=False, encoding='utf-8-sig')
required_inventory

## Section 2: 当前数据全面体检

In [ ]:
def pandas_readable(path: Path):
    try:
        if path.suffix.lower() == '.csv':
            pd.read_csv(path, nrows=5)
        else:
            pd.read_parquet(path)
        return True, ''
    except Exception as e:
        return False, f'{type(e).__name__}: {str(e)[:240]}'

def duck_source(path: Path):
    q = path.as_posix().replace("'", "''")
    return f"read_csv_auto('{q}')" if path.suffix.lower()=='.csv' else f"read_parquet('{q}')"

def audit_one(path: Path):
    row = {'file': path.name, 'path': str(path.relative_to(PROJECT_ROOT)), 'exists': path.exists(), 'size_mb': path.stat().st_size/1024/1024}
    ok, err = pandas_readable(path)
    row['pandas_readable'] = ok; row['pandas_error'] = err
    try:
        src = duck_source(path)
        desc = duckdb.sql(f'describe select * from {src}').df()
        cols = desc['column_name'].tolist()
        row['duckdb_readable'] = True; row['duckdb_error'] = ''; row['columns'] = ';'.join(cols)
        row['rows'] = duckdb.sql(f'select count(*) from {src}').fetchone()[0]
        for c in ['date','report_date','ann_date','publish_date','list_date','delist_date']:
            if c in cols:
                mn, mx = duckdb.sql(f'select min(try_cast({c} as date)), max(try_cast({c} as date)) from {src}').fetchone()
                row[f'{c}_min'] = mn; row[f'{c}_max'] = mx
        for c in ['code','index_code','tracking_index','industry']:
            if c in cols:
                row[f'n_{c}'] = duckdb.sql(f'select count(distinct {c}) from {src} where {c} is not null').fetchone()[0]
        key_cols = [c for c in cols if c in ['close','close_adj','total_mv','float_mv','turnover','amount','volume','industry','weight','ann_date','publish_date','revenue','net_profit','eps_ttm','tracking_index','nav','value']]
        if row['rows'] and key_cols:
            expr = ', '.join([f"avg(case when {c} is not null then 1.0 else 0.0 end) as {c}" for c in key_cols])
            row['nonnull_rate_json'] = json.dumps(duckdb.sql(f'select {expr} from {src}').df().iloc[0].round(4).to_dict(), ensure_ascii=False)
    except Exception as e:
        row['duckdb_readable'] = False; row['duckdb_error'] = f'{type(e).__name__}: {str(e)[:240]}'
    return row

files = sorted([p for p in RAW.rglob('*') if p.suffix.lower() in ['.parquet','.csv']])
audit_report = pd.DataFrame([audit_one(p) for p in files])
audit_report.to_csv(RAW / 'current_data_audit_report.csv', index=False, encoding='utf-8-sig')
audit_report.head(30)

## Section 3: 数据缺口矩阵

In [ ]:
gap = pd.read_csv(RAW / 'ifind_data_gap_matrix.csv')
gap.to_csv(RAW / 'ifind_data_gap_matrix.csv', index=False, encoding='utf-8-sig')
gap.sort_values(['priority','must_refetch','data_item']).head(50)

## Section 4: iFinD 通用接口封装

In [ ]:
FIELD_CANDIDATES = {
    'stock_daily': {
        'close': ['close','ths_close_price_stock','close_price'],
        'close_adj': ['ths_adj_close_stock','ths_close_price_stock_backward_adjusted','close_adj'],
        'return': ['ths_chg_ratio_stock','pct_chg','return'],
        'total_mv': ['ths_total_mv_stock','total_mv','market_cap'],
        'float_mv': ['ths_float_mv_stock','float_mv','circulating_market_cap'],
        'turnover': ['ths_turnover_ratio_stock','turnover'],
        'amount': ['ths_amount_stock','amount'],
        'volume': ['ths_vol_stock','volume'],
        'list_status': ['ths_listing_status_stock','list_status'],
        'is_st': ['ths_st_stock','is_st'],
        'suspend_flag': ['ths_suspend_stock','suspend_flag'],
        'limit_up_down_flag': ['ths_limit_status_stock','limit_up_down_flag'],
    },
    'financials': {
        'ann_date': ['ths_regular_report_actual_dd_stock','ann_date','publish_date'],
        'revenue': ['ths_operating_revenue_stock','revenue'],
        'net_profit': ['ths_np_parent_company_owners_stock','net_profit'],
        'net_profit_ttm': ['ths_np_parent_company_owners_ttm_stock','net_profit_ttm'],
        'eps': ['ths_basic_eps_stock','eps'],
        'eps_ttm': ['ths_eps_ttm_stock','eps_ttm'],
        'bps': ['ths_bps_stock','bps'],
        'total_assets': ['ths_total_assets_stock','total_assets'],
        'total_equity': ['ths_total_equity_stock','total_equity'],
        'roa': ['ths_roa_stock','roa'],
        'gpm': ['ths_gross_profit_margin_stock','gpm'],
        'ato': ['ths_total_asset_turnover_ttm_stock','ato'],
        'dtoa': ['ths_asset_liab_ratio_stock','dtoa'],
        'dividend_ttm': ['ths_cash_dividend_ttm_stock','dividend_ttm'],
        'div_yield': ['ths_dividend_yield_stock','div_yield'],
    }
}

failure_rows = []
def log_failure(task, key, error):
    failure_rows.append({'time': datetime.now().isoformat(timespec='seconds'), 'task': task, 'key': str(key), 'error': str(error)[:1000]})
    pd.DataFrame(failure_rows).to_csv(FAIL_LOG, index=False, encoding='utf-8-sig')

def retry_call(task, key, fn, *args, **kwargs):
    last = None
    for i in range(MAX_RETRIES):
        try:
            out = fn(*args, **kwargs)
            time.sleep(RATE_LIMIT_SLEEP)
            return out
        except Exception as e:
            last = e
            time.sleep(RATE_LIMIT_SLEEP * (2 ** i))
    log_failure(task, key, last)
    return None

def save_part(df, part_path):
    part_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(part_path, index=False)
    return part_path

def concat_parts(pattern, out_path):
    parts = sorted(Path().glob(pattern)) if isinstance(pattern, str) else sorted(pattern)
    dfs = [pd.read_parquet(p) for p in parts if p.exists()]
    if not dfs:
        return pd.DataFrame()
    df = pd.concat(dfs, ignore_index=True)
    df.to_parquet(out_path, index=False)
    return df

def ensure_ifind_ready():
    if not IFIND_LOGGED_IN:
        raise RuntimeError('iFinD 未登录。请设置账号环境变量并确认权限后重跑。')

def call_basic(codes, fields, params=''):
    ensure_ifind_ready()
    return retry_call('basic', f'{len(codes)}:{fields}', THS_BasicData, ','.join(codes), ';'.join(fields), params)

def call_hq(codes, fields, start, end, params=''):
    ensure_ifind_ready()
    return retry_call('hq', f'{len(codes)}:{fields}:{start}:{end}', THS_HQ, ','.join(codes), ';'.join(fields), params, start, end)

def call_edb(indicator, start, end):
    ensure_ifind_ready()
    return retry_call('edb', indicator, THS_EDB, indicator, start, end)

def ifind_to_df(resp):
    if resp is None:
        return pd.DataFrame()
    if hasattr(resp, 'data'):
        data = resp.data
        if isinstance(data, pd.DataFrame):
            return data.copy()
    if isinstance(resp, pd.DataFrame):
        return resp.copy()
    raise ValueError(f'无法识别 iFinD 返回对象: {type(resp)}')

## Section 5: 补齐个股日频数据

In [ ]:
def fetch_stock_daily_candidate(stock_codes):
    out = RAW / 'stock_daily_ifind_candidate.parquet'
    if not FETCH_STOCK_DAILY:
        return pd.DataFrame()
    fields = ['close','close_adj','return','total_mv','float_mv','turnover','amount','volume','list_status','is_st','suspend_flag','limit_up_down_flag']
    for i in range(0, len(stock_codes), STOCK_BATCH_SIZE):
        batch = stock_codes[i:i+STOCK_BATCH_SIZE]
        part = PART_DIR / 'stock_daily' / f'part_{i//STOCK_BATCH_SIZE:05d}.parquet'
        if part.exists():
            continue
        # TODO: 将 FIELD_CANDIDATES 探测到的实际 iFinD 字段名传入 call_hq。
        resp = call_hq(batch, fields, START_DATE, END_DATE, params='Fill:Previous')
        df = ifind_to_df(resp)
        df['source'] = 'ifind'
        save_part(df, part)
    return concat_parts((PART_DIR/'stock_daily').glob('*.parquet'), out)

## Section 6: 补齐历史行业分类

In [ ]:
def fetch_stock_industry_candidate(stock_codes, month_ends, scheme='CITIC1'):
    out = RAW / 'stock_industry_ifind_candidate.parquet'
    if not FETCH_STOCK_INDUSTRY:
        return pd.DataFrame()
    for d in month_ends:
        part = PART_DIR / 'stock_industry' / f'industry_{pd.Timestamp(d).date()}.parquet'
        if part.exists():
            continue
        rows = []
        for i in range(0, len(stock_codes), STOCK_BATCH_SIZE):
            batch = stock_codes[i:i+STOCK_BATCH_SIZE]
            # TODO: 确认 iFinD 历史行业接口/字段；必须以 d 为历史时点。
            resp = call_basic(batch, ['industry_code','industry_name'], params=f'TradeDate={pd.Timestamp(d):%Y-%m-%d};Industry={scheme}')
            df = ifind_to_df(resp)
            df['date'] = pd.Timestamp(d); df['industry_scheme'] = scheme; df['source'] = 'ifind'; df['pit_quality'] = 'historical_snapshot'
            rows.append(df)
        save_part(pd.concat(rows, ignore_index=True), part)
    return concat_parts((PART_DIR/'stock_industry').glob('*.parquet'), out)

## Section 7: 补齐历史指数成分股和权重

In [ ]:
def fetch_index_constituents_candidate(index_codes, month_ends):
    out = RAW / 'index_constituents_ifind_candidate.parquet'
    if not FETCH_INDEX_CONSTITUENTS:
        return pd.DataFrame()
    for idx in index_codes:
        for d in month_ends:
            part = PART_DIR / 'index_constituents' / f'{idx.replace(".", "_")}_{pd.Timestamp(d):%Y-%m-%d}.parquet'
            if part.exists():
                continue
            # TODO: 确认 iFinD 成分权重接口名称和字段。必须返回历史 date 的成分与 weight。
            resp = retry_call('index_constituents', f'{idx}:{d}', THS_DR, 'index_constituents_weight', f'IndexCode={idx};EndDate={pd.Timestamp(d):%Y-%m-%d}') if IFIND_LOGGED_IN else None
            df = ifind_to_df(resp) if resp is not None else pd.DataFrame()
            if not df.empty:
                df['date'] = pd.Timestamp(d); df['index_code'] = idx; df['source'] = 'ifind'; df['snapshot_quality'] = 'historical_weight'
            else:
                log_failure('index_constituents', f'{idx}:{d}', 'empty response or interface not confirmed')
            save_part(df, part)
    return concat_parts((PART_DIR/'index_constituents').glob('*.parquet'), out)

## Section 8: 补齐财务 PIT 数据

In [ ]:
def fetch_financials_candidate(stock_codes, report_dates):
    out = RAW / 'stock_financials_ifind_candidate.parquet'
    if not FETCH_FINANCIALS:
        return pd.DataFrame()
    fields = list(FIELD_CANDIDATES['financials'].keys())
    for rd in report_dates:
        for i in range(0, len(stock_codes), STOCK_BATCH_SIZE):
            part = PART_DIR / 'stock_financials' / f'fin_{pd.Timestamp(rd):%Y-%m-%d}_{i//STOCK_BATCH_SIZE:05d}.parquet'
            if part.exists():
                continue
            resp = call_basic(stock_codes[i:i+STOCK_BATCH_SIZE], fields, params=f'ReportDate={pd.Timestamp(rd):%Y-%m-%d};ReportType=1')
            df = ifind_to_df(resp)
            df['report_date'] = pd.Timestamp(rd); df['source'] = 'ifind'; df['pit_quality'] = np.where(df.get('ann_date', pd.Series(index=df.index)).notna(), 'strict_ann_date', 'needs_review')
            save_part(df, part)
    return concat_parts((PART_DIR/'stock_financials').glob('*.parquet'), out)

## Section 9: 补齐 ETF 信息和 ETF 行情

In [ ]:
def fetch_etf_candidates(etf_codes):
    if not FETCH_ETF:
        return pd.DataFrame(), pd.DataFrame()
    info_out = RAW / 'etf_info_ifind_candidate.parquet'
    daily_out = RAW / 'etf_daily_ifind_candidate.parquet'
    info_fields = ['name','list_date','delist_date','tracking_index','fund_type','manager']
    daily_fields = ['nav','close','close_adj','amount','volume','turnover','premium_discount']
    info = ifind_to_df(call_basic(etf_codes, info_fields, params=f'TradeDate={END_DATE}'))
    info['source'] = 'ifind'
    info.to_parquet(info_out, index=False)
    for i in range(0, len(etf_codes), STOCK_BATCH_SIZE):
        part = PART_DIR / 'etf_daily' / f'etf_daily_{i//STOCK_BATCH_SIZE:05d}.parquet'
        if part.exists():
            continue
        df = ifind_to_df(call_hq(etf_codes[i:i+STOCK_BATCH_SIZE], daily_fields, START_DATE, END_DATE))
        df['source'] = 'ifind'
        save_part(df, part)
    daily = concat_parts((PART_DIR/'etf_daily').glob('*.parquet'), daily_out)
    return info, daily

## Section 10: 补齐宏观指标

In [ ]:
macro_plan = required_inventory[required_inventory['data_item'].str.startswith('macro_')].copy()
def safe_indicator_name(name):
    return ''.join(ch if ch.isalnum() or ch in ['_','-'] else '_' for ch in name)

def fetch_macro_candidates(macro_code_map):
    reports = []
    for _, row in macro_plan.iterrows():
        name = row['data_item'].replace('macro_', '')
        code = macro_code_map.get(name)
        out = RAW / f'macro_{safe_indicator_name(name)}_ifind_candidate.parquet'
        if not code:
            log_failure('macro', name, 'missing iFinD/EDB code mapping')
            reports.append({'indicator_name': name, 'status': 'missing_code'})
            continue
        if out.exists():
            df = pd.read_parquet(out)
        else:
            df = ifind_to_df(call_edb(code, START_DATE, END_DATE))
            df['indicator_name'] = name; df['source'] = 'ifind'; df['pit_quality'] = 'release_calendar_unverified'
            df.to_parquet(out, index=False)
        reports.append({'indicator_name': name, 'rows': len(df), 'date_min': pd.to_datetime(df.get('date')).min() if 'date' in df else pd.NaT, 'date_max': pd.to_datetime(df.get('date')).max() if 'date' in df else pd.NaT, 'nonnull_rate': df.get('value', pd.Series(dtype=float)).notna().mean() if 'value' in df else np.nan})
    rep = pd.DataFrame(reports)
    rep.to_csv(RAW / 'macro_ifind_coverage_report.csv', index=False, encoding='utf-8-sig')
    return rep

# 示例：请按 iFinD 实际宏观代码补全。轻纺城成交量若无权限/无代码，使用人工 CSV 通道。
MACRO_CODE_MAP = {}

## Section 11: 候选数据校验

In [ ]:
def validate_candidate(path, required_fields, key_fields, start=START_DATE, end=END_DATE):
    row = {'candidate_file': str(path.relative_to(PROJECT_ROOT)), 'exists': path.exists()}
    if not path.exists():
        row['passed'] = False; row['notes'] = 'missing'; return row
    try:
        df = pd.read_parquet(path)
        row['pandas_readable'] = True; row['rows'] = len(df); row['columns'] = ';'.join(df.columns)
        missing = [c for c in required_fields if c not in df.columns]
        row['missing_fields'] = ';'.join(missing)
        if 'date' in df.columns:
            dt = pd.to_datetime(df['date'], errors='coerce')
            row['date_min'] = dt.min(); row['date_max'] = dt.max()
            row['date_coverage_ok'] = (dt.min() <= pd.Timestamp(start)) and (dt.max() >= pd.Timestamp(end))
        if key_fields and all(k in df.columns for k in key_fields):
            row['duplicate_key_rows'] = int(df.duplicated(key_fields).sum())
        nn = {c: round(float(df[c].notna().mean()),4) for c in required_fields if c in df.columns}
        row['nonnull_rate_json'] = json.dumps(nn, ensure_ascii=False)
        row['passed'] = (not missing) and row.get('duplicate_key_rows', 0) == 0
        row['notes'] = ''
    except Exception as e:
        row['pandas_readable'] = False; row['passed'] = False; row['notes'] = repr(e)
    return row

validation_plan = {
    'stock_daily_ifind_candidate.parquet': (['date','code','close','close_adj','return','total_mv','float_mv','turnover','amount','volume','list_status','is_st','suspend_flag','limit_up_down_flag'], ['date','code']),
    'stock_industry_ifind_candidate.parquet': (['date','code','industry_code','industry_name','industry_scheme','source','pit_quality'], ['date','code','industry_scheme']),
    'index_constituents_ifind_candidate.parquet': (['date','index_code','code','weight','source','snapshot_quality'], ['date','index_code','code']),
    'stock_financials_ifind_candidate.parquet': (['code','report_date','ann_date','publish_date','revenue','net_profit','net_profit_ttm','eps','eps_ttm','bps','total_assets','total_equity','roa','gpm','ato','dtoa','dividend_ttm','div_yield','source','pit_quality'], ['code','report_date']),
    'etf_info_ifind_candidate.parquet': (['code','name','list_date','delist_date','tracking_index','fund_type','manager'], ['code']),
    'etf_daily_ifind_candidate.parquet': (['date','code','nav','close','close_adj','amount','volume','turnover','premium_discount'], ['date','code']),
}
validation = pd.DataFrame([validate_candidate(RAW / fn, req, keys) for fn,(req,keys) in validation_plan.items()])
validation.to_csv(RAW / 'ifind_candidate_validation_report.csv', index=False, encoding='utf-8-sig')
validation

## Section 12: 是否覆盖正式 raw 文件

In [ ]:
def promote_candidate(candidate_name, canonical_name, validation_df):
    cand = RAW / candidate_name
    canon = RAW / canonical_name
    rec = validation_df[validation_df['candidate_file'].str.endswith(candidate_name)]
    ok = (not rec.empty) and bool(rec.iloc[0]['passed'])
    if not OVERWRITE_CANONICAL:
        print(f'SKIP {canonical_name}: OVERWRITE_CANONICAL=False')
        return False
    if not ok:
        print(f'SKIP {canonical_name}: candidate validation failed')
        return False
    if canon.exists():
        backup = canon.with_suffix(canon.suffix + f'.bak_{datetime.now():%Y%m%d_%H%M%S}')
        shutil.copy2(canon, backup)
        print('backup:', backup)
    shutil.copy2(cand, canon)
    print('promoted:', cand, '->', canon)
    return True

promotion_map = {
    'stock_daily_ifind_candidate.parquet': 'stock_daily.parquet',
    'stock_industry_ifind_candidate.parquet': 'stock_industry.parquet',
    'index_constituents_ifind_candidate.parquet': 'index_constituents.parquet',
    'stock_financials_ifind_candidate.parquet': 'stock_financials.parquet',
    'etf_info_ifind_candidate.parquet': 'etf_info.parquet',
    'etf_daily_ifind_candidate.parquet': 'etf_daily.parquet',
}
for cand, canon in promotion_map.items():
    promote_candidate(cand, canon, validation)

## Section 13: 最终结论

In [ ]:
def build_readiness(gap, validation):
    rows = []
    for _, g in gap.iterrows():
        cand_name = Path(str(g['candidate_file'])).name
        v = validation[validation['candidate_file'].str.endswith(cand_name)] if not validation.empty and 'candidate_file' in validation else pd.DataFrame()
        candidate_passed = (not v.empty) and bool(v.iloc[0].get('passed', False))
        if candidate_passed and 'strict' in str(g['pit_status']):
            readiness = '候选通过，仍需人工确认 PIT 字段口径'
        elif candidate_passed:
            readiness = '候选通过校验，可进入人工口径复核'
        elif g['must_refetch'] == 'yes':
            readiness = '仍需 iFinD 补齐或字段调整'
        else:
            readiness = '当前数据可部分复用，建议统一重写为 pandas/pyarrow 可读 parquet'
        rows.append({**g.to_dict(), 'candidate_validation_passed': candidate_passed, 'readiness_conclusion': readiness})
    out = pd.DataFrame(rows)
    out.to_csv(RAW / 'ifind_final_data_readiness_report.csv', index=False, encoding='utf-8-sig')
    return out

final_readiness = build_readiness(gap, validation)
final_readiness[['data_item','can_reuse','must_refetch','pit_status','candidate_validation_passed','readiness_conclusion']]